In [4]:
import pandas as pd
import plotly.express as px
from pathlib import Path

RAW = Path("../data/raw")
wages = pd.read_excel(next(RAW.glob("entgelte_berufe*.xlsx")), header=8).iloc[:, :6]
wages.columns = ["group", "Insgesamt", "Helfer", "Fachkräfte", "Spezialisten", "Experten"]
wages = wages[wages["group"].astype(str).str.match(r"^\d{3} ")].copy()   # только строки с кодом группы
wages["code3"] = wages["group"].str[:3]
levels = ["Fachkräfte", "Spezialisten", "Experten"]
for c in levels:
    # "> 8.050" = выше потолка соцстрахования, ставим 8050; "x" = нет данных
    wages[c] = pd.to_numeric(wages[c].replace({"> 8.050": 8050}), errors="coerce")
wages_long = wages.melt(id_vars=["group", "code3"], value_vars=levels,
                        var_name="level", value_name="wage").dropna()
wages_long.head()

,group,code3,level,wage
0,111 Landwirtschaft,111,Fachkräfte,2963.0
1,112 Tierwirtschaft,112,Fachkräfte,2725.0
2,113 Pferdewirtschaft,113,Fachkräfte,2572.0
3,114 Fischwirtschaft,114,Fachkräfte,3042.0
4,115 Tierpflege,115,Fachkräfte,3170.0


In [7]:
epa = pd.read_csv(RAW / "epa_2025.csv", sep=";", decimal=",", encoding="utf-8-sig",
                  na_values=["X", " "], dtype={"Beruf.BA_SCHL": str})
epa25 = epa[(epa["Jahr.DESC"] == 2025) & (epa["Bundesland RZ.BA_SCHL"] == "d")
            & (epa["Beruf.BA_SCHL"].str.len() == 4)].copy()
epa25["code3"] = epa25["Beruf.BA_SCHL"].str[:3]
epa25["level"] = epa25["Anforderungsniveau.DESC"]

short = epa25.groupby(["code3", "level"]).agg(
    rated=("Einstufung", "size"),
    n_shortage=("Einstufung", lambda x: (x == "Engpassberuf").sum())).reset_index()
short["status"] = short["n_shortage"].gt(0).map({True: "shortage", False: "no shortage"})

wm = wages_long.merge(short, on=["code3", "level"])
wm.groupby(["level", "status"])["wage"].median().unstack().round(0)

status,no shortage,shortage
level,,
Experten,5889.0,6114.0
Fachkräfte,3932.0,3763.0
Spezialisten,5047.0,4632.0


In [8]:
fig = px.box(wm, x="level", y="wage", color="status", points="all", hover_name="group",
             category_orders={"level": levels, "status": ["shortage", "no shortage"]},
             color_discrete_map={"shortage": "#d62728", "no shortage": "#1f77b4"}, height=600,
             title="Do shortage occupations pay less? Median monthly gross wage by occupation group, Dec 2025",
             labels={"wage": "Median gross monthly wage, €", "level": "Skill level", "status": ""})
fig.show()
fig.write_html("../output/charts/04_wages_shortage.html")

## Finding 4: Shortage occupations pay slightly less, but wages there rise faster

- Median gross monthly wage, shortage vs. no shortage: Fachkräfte 3,763 € vs 3,932 € (−4%), Spezialisten 4,632 € vs 5,047 € (−8%), Experten 6,114 € vs 5,889 € (+4%).
- Lowest-paid shortage groups: food retail (2,616 €), gastronomy (2,671 €).
- Exception: nursing pays above average (4,581 €) and is still short → working conditions matter, not only pay.
- Wages in shortage occupations grew faster (median +14.2% vs +10.4%), but wage growth is itself one of the BA shortage indicators, so this link is partly built in.
- H4 is **partly supported**: the effect exists but is weak.

Sources: BA Entgeltatlas (Dec 2025), BA Engpassanalyse 2025. Own calculations.
